# Activity 11: Logistic Regression with TensorFlow / Keras

## Learning Objectives
By the end of this activity you will be able to:
- Build a logistic regression model using `tf.keras` with a single dense layer and sigmoid activation
- Understand what binary cross-entropy means in Keras and how it maps to math from Activity 10
- Use `EarlyStopping` and `ModelCheckpoint` callbacks to prevent over-training
- Plot training vs validation loss curves and interpret them confidently
- Compare TensorFlow model performance against sklearn baseline

---
> **Why TensorFlow for logistic regression?**  
> Logistic regression IS a 1-layer neural net. Understanding it in Keras makes the jump to deep learning trivial — you already know the framework.

## Part 1 — Imports & Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")

# --- Dataset ---
# WHY 20 features?  More realistic than 2 — forces TF to show its strengths.
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=15,
    n_redundant=5,
    random_state=42
)
y = y.astype(np.float32)   # Keras expects float32 for binary target

scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val,  y_train, y_val  = train_test_split(X_train, y_train, test_size=0.15, random_state=42)

print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")

## Part 2 — Architecture

Logistic regression in Keras = **Dense(1, activation='sigmoid')**.  
No hidden layers. One weight per feature, one bias.

```
Input (20,)  →  Dense(1, sigmoid)  →  ŷ ∈ (0,1)
```

In [ ]:
def build_logistic_model(input_dim, learning_rate=0.01):
    """
    WHY Sequential? Logistic regression is a straight pipeline — no branches.
    COMMON ERROR: Using softmax instead of sigmoid for binary classification.
                  Softmax gives two outputs; sigmoid gives one probability.
    """
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(1, activation='sigmoid',
                     kernel_initializer='glorot_uniform',
                     name='logistic_output')
    ], name='LogisticRegression')

    # WHY 'binary_crossentropy'?  Maps exactly to the log-loss we derived from scratch.
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        # WHY accuracy metric here?  Binary accuracy is just threshold(ŷ, 0.5).
        metrics=['accuracy']
    )
    return model

model = build_logistic_model(input_dim=X_train.shape[1])
model.summary()

## Part 3 — Callbacks

| Callback | Purpose |
|---|---|
| `EarlyStopping` | Stop training when val_loss stops improving — prevents overfitting |
| `ModelCheckpoint` | Save the *best* weights, not the last weights |
| `ReduceLROnPlateau` | Automatically lower LR when stuck in a plateau |

In [ ]:
# WHY patience=15?  Gives the model enough time to escape a plateau
# before we give up.  Too small → quit too early.
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,   # reverts to the lowest val_loss checkpoint
    verbose=1
)

checkpoint = callbacks.ModelCheckpoint(
    filepath='/tmp/best_logistic_tf.weights.h5',
    monitor='val_loss',
    save_best_only=True,
    save_weights_only=True,
    verbose=0
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,          # halve LR on plateau
    patience=7,
    min_lr=1e-6,
    verbose=1
)

print("Callbacks ready.")

## Part 4 — Training

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, checkpoint, reduce_lr],
    verbose=0   # silent; we'll plot the history
)

actual_epochs = len(history.history['loss'])
print(f"Training stopped at epoch {actual_epochs}")

## Part 5 — Loss & Accuracy Curves

In [ ]:
hist = history.history
epochs_range = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Loss
axes[0].plot(epochs_range, hist['loss'],     label='Train Loss',     color='steelblue')
axes[0].plot(epochs_range, hist['val_loss'], label='Val Loss',       color='crimson', linestyle='--')
axes[0].set_title('Binary Cross-Entropy Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, hist['accuracy'],     label='Train Accuracy', color='steelblue')
axes[1].plot(epochs_range, hist['val_accuracy'], label='Val Accuracy',   color='crimson', linestyle='--')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("TensorFlow Logistic Regression — Training History", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Part 6 — Test Evaluation & sklearn Comparison

In [ ]:
# TF model evaluation
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
y_pred_tf = (model.predict(X_test, verbose=0).flatten() >= 0.5).astype(int)

# sklearn baseline
sk_model = LogisticRegression(max_iter=1000, random_state=42)
sk_model.fit(X_train, y_train)
y_pred_sk = sk_model.predict(X_test)
acc_sk = accuracy_score(y_test, y_pred_sk)

print("=" * 48)
print(f"  TensorFlow accuracy  : {test_acc:.4f}   loss: {test_loss:.4f}")
print(f"  sklearn accuracy     : {acc_sk:.4f}")
print("=" * 48)

print("\n--- TF Classification Report ---")
print(classification_report(y_test, y_pred_tf))

## Part 7 — Inspect Learned Weights

In [ ]:
# WHY inspect weights?  They represent log-odds ratio — larger absolute value
# = stronger feature influence on the prediction.
weights, bias = model.get_layer('logistic_output').get_weights()
weights_flat = weights.flatten()

feature_names = [f"Feature {i+1}" for i in range(X_train.shape[1])]
importance = pd.DataFrame({'Feature': feature_names, 'Weight': weights_flat})
importance = importance.reindex(importance['Weight'].abs().sort_values(ascending=False).index)

plt.figure(figsize=(10, 6))
colors = ['steelblue' if w > 0 else 'crimson' for w in importance['Weight']]
plt.barh(importance['Feature'], importance['Weight'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Weight Value (Log-Odds Contribution)')
plt.title('Learned Feature Weights — TF Logistic Regression')
plt.tight_layout()
plt.show()

print(f"Bias (intercept): {bias[0]:.4f}")

## Summary

| Component | Code |
|---|---|
| Architecture | `Dense(1, activation='sigmoid')` |
| Loss | `binary_crossentropy` |
| Optimiser | `Adam` |
| Regularisation via training | `EarlyStopping(restore_best_weights=True)` |

**Key takeaways:**
- Logistic regression is literally a 1-neuron neural network — learning Keras here pays dividends for deep learning
- `EarlyStopping(restore_best_weights=True)` is superior to training for a fixed number of epochs
- The learned weights are interpretable: their magnitude = feature importance, sign = direction

**Next:** Activity 12 — Logistic Regression in PyTorch